In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import glob, os
import wrf
from netCDF4 import Dataset

In [ ]:
# ── Simulation definitions ─────────────────────────────────────────────────────
base_111m = '/glade/derecho/scratch/gduine/mountain_fire/111m/'
base_1km  = '/glade/derecho/scratch/gduine/mountain_fire/1km/'

simulations = [
    {'label': 'No fire (ref)',  'dir': base_111m + 'ifire0/ref/',       'domain': 'd04', 'color': 'gray',   'lw': 1.5, 'ls': '-'},
    {'label': 'z0 double',      'dir': base_111m + 'ifire0/z0_double/', 'domain': 'd04', 'color': 'blue',   'lw': 1.5, 'ls': '--'},
    {'label': 'Fire (ifire2)',   'dir': base_111m + 'ifire2/',           'domain': 'd04', 'color': 'red',    'lw': 1.5, 'ls': '-'},
    {'label': '1 km',           'dir': base_1km,                         'domain': 'd03', 'color': 'green',  'lw': 1.5, 'ls': '-'},
]

# ── Station locations ──────────────────────────────────────────────────────────
stations = {
    'START':       {'lat': 34.318,    'lon': -118.968},
    'SPOT':        {'lat': 34.2528,   'lon': -119.0284},
    'Spot Valley': {'lat': 34.271191, 'lon': -119.015999},
}

out_dir = '.'

In [ ]:
# ── Extract U10/V10 time series at all stations for one simulation ─────────────
def extract_timeseries(sim, stations):
    domain  = sim['domain']
    wrf_dir = sim['dir']

    files = sorted(glob.glob(os.path.join(wrf_dir, f'wrfout_{domain}_*')))
    if not files:
        print(f'  No files found in {wrf_dir}')
        return None
    print(f'  {sim["label"]}: {len(files)} files')

    # Grid indices for each station (from first file)
    with Dataset(files[0]) as nc0:
        lats_list = [s['lat'] for s in stations.values()]
        lons_list = [s['lon'] for s in stations.values()]
        xy_pts    = wrf.ll_to_xy(nc0, lats_list, lons_list)

    times  = []
    u10_ts = {name: [] for name in stations}
    v10_ts = {name: [] for name in stations}

    for fName in files:
        nc      = Dataset(fName)
        n_times = nc.dimensions['Time'].size

        for idt in range(n_times):
            timeWRF = wrf.getvar(nc, 'Times', idt)
            times.append(pd.to_datetime(wrf.to_np(timeWRF)))

            U10_np = wrf.to_np(wrf.getvar(nc, 'U10', idt))
            V10_np = wrf.to_np(wrf.getvar(nc, 'V10', idt))

            for i, name in enumerate(stations):
                ix = int(xy_pts[0, i])   # column index
                iy = int(xy_pts[1, i])   # row index
                u10_ts[name].append(float(U10_np[iy, ix]))
                v10_ts[name].append(float(V10_np[iy, ix]))

        nc.close()

    # Convert to PST
    times = pd.DatetimeIndex(times) - pd.Timedelta(hours=8)

    result = {'times': times}
    for name in stations:
        u = np.array(u10_ts[name])
        v = np.array(v10_ts[name])
        result[name] = {
            'wspd': np.sqrt(u**2 + v**2),
            'wdir': (180 + np.degrees(np.arctan2(u, v))) % 360,
        }

    return result


# ── Run extraction for all simulations ────────────────────────────────────────
all_data = {}
for sim in simulations:
    print(f'Processing {sim["label"]} ...')
    all_data[sim['label']] = extract_timeseries(sim, stations)

In [ ]:
# ── Plot ───────────────────────────────────────────────────────────────────────
n_stations = len(stations)
fig, axes  = plt.subplots(2, n_stations, figsize=(6 * n_stations, 9), sharex=True)

for j, st_name in enumerate(stations):
    ax_spd = axes[0, j]
    ax_dir = axes[1, j]

    for sim in simulations:
        data = all_data[sim['label']]
        if data is None:
            continue
        times = data['times']
        wspd  = data[st_name]['wspd']
        wdir  = data[st_name]['wdir']

        ax_spd.plot(times, wspd,
                    color=sim['color'], lw=sim['lw'], ls=sim['ls'],
                    label=sim['label'])
        # scatter for direction avoids spurious lines at the 360°→0° wrap
        ax_dir.scatter(times, wdir,
                       color=sim['color'], s=6, label=sim['label'])

    # --- wind speed panel ---
    ax_spd.set_title(st_name, fontsize=15, fontweight='bold')
    ax_spd.set_ylabel('Wind speed [m s$^{-1}$]', fontsize=13)
    ax_spd.tick_params(labelsize=12)
    ax_spd.grid(True, alpha=0.4)
    if j == 0:
        ax_spd.legend(fontsize=11, loc='upper left')

    # --- wind direction panel ---
    ax_dir.set_ylabel('Wind direction', fontsize=13)
    ax_dir.set_ylim(0, 360)
    ax_dir.set_yticks([0, 90, 180, 270, 360])
    ax_dir.set_yticklabels(['N (0°)', 'E (90°)', 'S (180°)', 'W (270°)', 'N (360°)'])
    ax_dir.tick_params(labelsize=12)
    ax_dir.grid(True, alpha=0.4)
    ax_dir.set_xlabel('Time (PST)', fontsize=13)
    ax_dir.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d\n%H:%M'))
    ax_dir.xaxis.set_major_locator(mdates.HourLocator(interval=6))

plt.tight_layout()
fig.savefig(os.path.join(out_dir, 'timeseries_winds_4sims.jpg'),
            dpi=120, bbox_inches='tight')
plt.show()
print('Done!')